# Intro til Machine Learning — 2: Aktiveringsfunktioner

I notebook 1 var ReLU og sigmoid "krølningen mellem lagene". Nu skal de i rampelyset,
for aktiveringsfunktionerne er dét, der overhovedet gør neurale netværk til andet end
lineær regression i festtøj.

I denne notebook:
1. **De fem store** — Sigmoid, Tanh, ReLU, Leaky ReLU og Softmax: formler, grafer og gradienter (afsnit 1)
2. **Aktiveringer i kamp** — vi træner netværk på måne-formede data og ser forskellen (afsnit 2)

> **Om opgaverne:** Der er med vilje flere opgaver, end du kan nå — du behøver ikke nå alt. Opgaver mærket **(find fejlen)** har en bevidst fejl, som du skal finde og rette (så en fejl dér er meningen). Nederst i notebooken ligger et par **ekstra opgaver**, hvis du får lyst til mere.
>
> Noget af det her er nyt og kan føles udfordrende i starten — og det er i virkeligheden ret normalt. Os faglige har selv stået i jeres sko da vi først startede. Vi vil prøve at forklare hvert skridt så klart som vi nu kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid, og tag altid gerne fat i en faglig hvis du får brug for det. 

## Setup

In [ ]:
# Plottehjælperen fra GitHub (Plan B: upload filen manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, make_blobs

from helpers import plot_decision_boundary

torch.manual_seed(42)
np.random.seed(42)

# 1: De fem store

Kort om *hvorfor*: to lineære lag uden noget imellem
er stadig ét lineært lag: lineært efter lineært er stadig lineært.
Uden noget ikke-lineært mellem lagene ender selv verdens dybeste netværk som én stor
`nn.Linear`. Aktiveringsfunktionen er den lille bøjning, der — gentaget gennem mange lag —
lader netværket tegne vilkårligt krøllede sammenhænge.

Vi bruger altså aktiveringsfunktioner for at tilegne vores neurale netværk en bedre evne til at udføre dens gæt.
Det er derfor også vigtigt at vælge den rigtige aktiveringsfunktion til det rigtige sammenhæng.
F.eks. er det nok en god ide at bruge en aktiveringsfunktion der sikrer at alle vores værdier ikke skalerer uendeligt stort, hvis vi gerne vil udregne en sandsynlighed (eg. et netværk der udregner sandsynligheden for at et givet ct scan viser en tumor). Måske vil vi derimod gerne brug noget andet til hvis vores netværk skal klassificere (eg. et netværk der bestemmer hvilket dyr et billede viser), måske noget tredje hvis den skal lave en approksimation (eg. et netværk der approximerer bevægelsen af partikler for at lave simulationer i fysik).

Overordnet set findes der 5 aktiveringsfunktioner som kan anses at skelne mellem de mest typiske sammenhæng. Disse er følgende:

| Funktion | Formel | Output-interval | Typisk brug |
|---|---|---|---|
| **Sigmoid** | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $(0, 1)$ | output ved **binær klassifikation** |
| **Tanh** | $\tanh(z)$ | $(-1, 1)$ | som sigmoid, men centreret om 0 |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | **standardvalget i skjulte lag** |
| **Leaky ReLU** | $z$ hvis $z>0$, ellers $0{,}01 z$ | $(-\infty, \infty)$ | ReLU med "lækage" — mere om hvorfor nedenfor |
| **Softmax** | $\dfrac{e^{z_i}}{\sum_j e^{z_j}}$ | sandsynligheder, sum = 1 | output ved **klassifikation med flere klasser** |

Lad os *se* dem. Her er en plotteskabelon — `torch.linspace` laver 200 jævnt fordelte
tal fra −5 til 5, og så plotter vi funktionen af dem:

In [ ]:
x = torch.linspace(-5, 5, 200)

plt.plot(x, torch.sigmoid(x))
plt.title("Sigmoid")
plt.xlabel("z")
plt.grid(True)
plt.show()

(De andre hedder `torch.tanh(x)`, `torch.relu(x)` og
`nn.functional.leaky_relu(x)` — dem skal I selv plotte i opgave 1.1.)

## Gradienterne — funktionernes skjulte personlighed

Under træning flyder gradienter **baglæns** gennem netværket (autograd i PyTorch), og de skal
*igennem* hver aktiveringsfunktion undervejs. Funktionens hældning bestemmer, hvor meget
gradient der slipper igennem.

Altså, ligesom hvordan vi skubbede på vores parametre indenfor regression til løbende at skabe en bedre model, gør vi det præcist samme i et neuralt netværk. Vi beregner altså baglæns gennem alle lagene for at vide hvordan vi skal skubbe på de forskellige neuroner i netværket for ende med en bedre loss funktion. Til ethvert tidspunkt ved vi ikke nødvendigvis de præcise værdier som gør vores netværk bedre- vi ved bare for alle neuronerne hvilken retning deres værdier skal bevæge sig. 

Det vigtigste at huske i alt det her er, at vi aldrig overordnet ved præcist hvor meget vi skal skubbe alle vægtene i vores netværk- kun hvilken retning. Vi ved heller ikke om den retning vi skubber vægtene vil føre til det absolut *bedste mulige* netværk- vi ved kun at det vil føre til et *bedre* netværk, eller i hvert fald en lavere resulterende loss.

I løbet af denne processe opstår 2 skjulte problemer som vi skal til at se på.

Vi kan bruge autograd til at plotte en funktions gradient uden at kende formlen for den:

In [ ]:
x = torch.linspace(-5, 5, 200, requires_grad=True)
y = torch.sigmoid(x)
y.sum().backward()          # giver gradienten i alle 200 punkter på én gang

plt.plot(x.detach(), y.detach(), label="sigmoid(z)")
plt.plot(x.detach(), x.grad, "--", label="gradient (hældning)")
plt.title("Sigmoid og dens gradient")
plt.legend()
plt.grid(True)
plt.show()

Se på den stiplede kurve: sigmoids gradient er **højst 0,25** — og ude i "halerne"
(store positive/negative z) er den praktisk talt **nul**. Sender man en gradient baglæns
gennem mange sigmoid-lag, ganges den med et lille tal for hvert lag og **forsvinder**. Dette er det første problem- det berømte *vanishing gradient*-problem, det at netværkets forreste lag lærer aldrig noget, da effektiviteten af træningen lag efter lag langsomt forsvinder.
(Du mærker det selv i ekstra-opgaven med et dybt sigmoid-netværk.)

ReLU har derimod det præcist modsatte problem: hældning præcis 1 for alle positive z (gradienten passerer urørt!) —
men hældning **0** for alle negative. En neuron, der altid får negative input, får aldrig
gradient og kan aldrig lære igen: en **død ReLU**. ReLU ender altså så småt med at gøre vores netværk sværre og sværre at træne ordentligt. 
Dette er faktisk hvorfor man opfandt **Leaky ReLU**,
som lader en lille smule gradient sive igennem på den negative side — et plaster på
problemet.

## Softmax: sandsynlighedsmaskinen

De fire første funktioner arbejder på ét tal ad gangen. **Softmax** er anderledes: den
tager en hel **stribe** tal (ét "point" pr. klasse) og laver dem om til sandsynligheder,
der er positive og summer til 1:

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])            # netværkets rå point for 3 klasser
probabilities = torch.softmax(point, dim=0)

print(probabilities)
print("sum:", probabilities.sum().item())

Klassen med flest point får den største sandsynlighed, men de andre får også en bid —
softmax siger ikke "vinderen tager alt", den siger *hvor sikker* den er.

### Opgaver

##### Opgave 1.1
Vi vil se de fire "almindelige" aktiveringsfunktioner ved siden af hinanden.

Prøv at tegne alle fire — sigmoid, tanh, ReLU og leaky ReLU — i ét 2×2-grid. Skabelonen har allerede lavet grid'et med `plt.subplots(2, 2)` og plottet sigmoid; udfyld de tre andre felter.

Aflæs for hver: hvad er output-intervallet?

Hint: `axes[0, 1]`, `axes[1, 0]` og `axes[1, 1]` er de tre tomme felter — brug `torch.tanh(x)`, `torch.relu(x)` og `nn.functional.leaky_relu(x)`.

In [ ]:
x = torch.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

# Det første felt er lavet for dig som forbillede:
axes[0, 0].plot(x, torch.sigmoid(x));  axes[0, 0].set_title("Sigmoid")

# Udfyld de tre andre — kun selve funktionen mangler i hvert:
axes[0, 1].plot(x, ...);  axes[0, 1].set_title("Tanh")        # <-- udfyld inde i denne
axes[1, 0].plot(x, ...);  axes[1, 0].set_title("ReLU")        # <-- udfyld inde i denne
axes[1, 1].plot(x, ...);  axes[1, 1].set_title("Leaky ReLU")  # <-- udfyld inde i denne

plt.tight_layout()
plt.show()

##### Opgave 1.2
Vi bygger sigmoid op fra bunden med `torch.exp`.

Formlen er $\sigma(z) = \dfrac{1}{1+e^{-z}}$. Prøv at udfylde den i koden, og tjek, at din udgave giver det samme som `torch.sigmoid`.

Hint: $e^{-z}$ skrives `torch.exp(-z)` i kode.

In [ ]:
def min_sigmoid(z):
    return ...      # ← udfyld beregningen af sigmoid heri

z = torch.tensor([-2.0, 0.0, 3.0])
print("min:    ", min_sigmoid(z))
print("PyTorch:", torch.sigmoid(z))

##### Opgave 1.3
Nu bygger du ReLU selv. ReLU sætter alle negative tal til 0 og lader positive tal passere urørt.

Prøv at udfylde funktionen — brug enten `torch.clamp(z, min=...)` (som klipper tal af ved en grænse så de ligger indenfor de givne grænser) eller `torch.where(betingelse, hvis_sand, hvis_falsk)`. Tjek mod PyTorch bagefter.

Hint: med clamp skal grænsen være 0, så alt under 0 klippes væk.

In [ ]:
def min_relu(z):
    return ...      # ← udfyld beregningen af ReLU heri

z = torch.tensor([-3.0, -0.5, 0.0, 2.0])
print("min:    ", min_relu(z))
print("PyTorch:", torch.relu(z))

##### Opgave 1.4
Vi ser nu på sigmoids gradient- det som i funktionens virkelige effekt under træning

Prøv at genbruge gradient-plotteskabelonen fra teorien ovenfor på sigmoid, og aflæs: hvad er gradienten cirka ved $z = \pm 5$?

Forestil dig nu en gradient, der under træning skal baglæns gennem **10 sigmoid-lag** og ganges med dette tal hver lag. Hvad sker der med netværke— og hvorfor er det et problem for træningen af den?

Hint: et lille tal (fx 0,2) ganget med sig selv ti gange bliver... hvor stort?

*(Tænk over det, og prøv måske at diskutére med din sidemand — I behøver ikke skrive svaret ned.)*

##### Opgave 1.5
Leaky ReLU har en indstillelig mængde "lækage" via `negative_slope` — hvor meget den lader værdier sive igennem på den negative side.

Prøv at plotte den med `negative_slope` på forskellige værdier som **0.01, 0.1 og 0.5** i samme figur (med labels og en legend). Hvornår holder den op med at ligne ReLU og kommer til at ligne en ret linje i stedet?

Hint: jo større `negative_slope`, jo mindre "knæk" ved 0- indtil en hvis overgang.

In [ ]:
x = torch.linspace(-5, 5, 200)
plt.plot(x, nn.functional.leaky_relu(x, negative_slope=0.01), label="Leaky ReLU") # ← lej rundt med negative_slope værdien
plt.legend()
plt.grid(True)
plt.show()

##### Opgave 1.6
Softmax omdanner en liste værdier til sandsynligheder, der summer til 1.

Prøv at udfylde softmax-kaldet (hvilken `dim`?) og tjek, at summen bliver 1. Gang derefter alle point med 10 (`point * 10`) og kør igen — hvad sker der med sandsynlighederne, og hvorfor giver det mening?

Hint: her er `point` en enkelt stribe tal (1D), så der skal normaliseres langs den eneste akse, `dim=0`.

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])
probabilities = torch.softmax(point, dim=...)   # ← udfyld: hvilken akse? point er 1D
print(probabilities, "| sum:", probabilities.sum().item())

##### Opgave 1.7 (find fejlen)
Her er softmax brugt på et batch med ét eksempel og tre klasser — men ALLE "sandsynligheder" bliver 1,0?!

Kig på tensorens shape `(1, 3)` og på `dim`-argumentet: hvilken retning bliver der normaliseret langs nu? Prøv at rette `dim`, så det giver mening.

Husk fra opgave 1.6: softmax skal summe hen over de tre klasser.

Hint: `dim=0` går ned ad kolonnerne (kun ét tal i hver → alt bliver 1). Hvilken dim går hen over de tre klasser?

In [ ]:
point = torch.tensor([[2.0, 1.0, 0.1]])     # shape (1, 3): 1 eksempel, 3 klasser
probabilities = torch.softmax(point, dim=0)
print(probabilities)                        # giver [[1., 1., 1.]] — det er vist ikke rigtige sandsynligheder

# 2: Aktiveringer i kamp — månedata

Nu skal påstandene testes. `make_moons` laver et syntetisk 2D-datasæt: to klasser formet
som halvmåner, der griber ind i hinanden. Det er perfekt til formålet, for med kun 2
features kan vi **tegne alt**, inklusive hvad netværket tænker:

In [ ]:
X_np, y_np = make_moons(n_samples=400, noise=0.2, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolors="black", s=25)
plt.title("To måner — kan et netværk skille dem ad?")
plt.show()

Ingen ret linje kan skille de to måner ad — prøv selv at "tegne" en med øjnene.
Spørgsmålet er, om et netværk kan tegne noget bedre.

Vi genbruger træningsopskriften fra afsnit 3 og pakker den i en funktion (så vi ikke
skal skrive de samme 10 linjer 10 gange — det er jo derfor, funktioner findes). Læs den
igennem: det er de samme fem trin som før:

In [ ]:
# train() ligger nu i helpers (samme 5-trins loop) — vi importerer den herfra
from helpers import train


In [ ]:
class MoonNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.activation = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

model = MoonNet()
history = train(model, X, y)
print("sluttab:", round(history[-1], 4))

Og nu det fede: `plot_decision_boundary` (fra hjælpefilen) farver **hele planen**
efter, hvad modellen ville svare i hvert punkt. Den sorte streg er netværkets
**beslutningsgrænse** — skillelinjen mellem "klasse 0" og "klasse 1":

In [ ]:
plot_decision_boundary(model, X, y, title="MoonNet med ReLU")

Prøv at se kurven- netværket har selv opfundet en bugtet grænse, der følger månerne.
Igen, dette (altså et neuralt netværk med flere lag) er kun muligt at bruge pga. aktiveringsfunktionerne. Dette er hvad aktiveringsfunktionerne giver os i den sidste ende.

### Opgaver

##### Opgave 2.1
Bare for virkelig at understrege pointen med aktiveringsfunktioner igen, kan i nu se hvad der sker hvis vi gør det samme som før uden dem.

Klassen nedenfor er MaaneNet **uden aktivering mellem lagene** (sigmoiden til sidst er kun for at få en sandsynlighed ud).

Prøv at træne den og plotte dens beslutningsgrænse. Sammenlign med ReLU-udgaven ovenfor. Overvej hvordan grænsen nu ser ud. Er der en årsag til at netværket ikke længere kan lære at genkende månerne, uanset hvor længe den træner?

Husk: uden noget ikke-lineært mellem lagene kan hele netværket skrives om til én stor `nn.Linear` — altså samtlige lag kan altid bare matematisk reduceres ned til ét enkelt lag.

In [ ]:
class LinearNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)          # ingen aktivering
        x = self.layer2(x)          # ingen aktivering
        x = self.sigmoid(self.layer3(x))
        return x

model_lin = LinearNet()
history = train(model_lin, X, y)
print("sluttab:", round(history[-1], 4))
plot_decision_boundary(model_lin, X, y, title="Uden aktivering")

##### Opgave 2.2
Nu skifter vi aktiveringsfunktion og ser hvilken der lærer at genkende månerne hurtigst under træning.

`FlexNet` tager aktiveringen som parameter. Prøv at træne den med **Sigmoid**, **Tanh** og **LeakyReLU** (én ad gangen) og plot de fire tabskurver i samme figur.

Hint: Aktiveringsfunktionerne eksisterer som `nn.Sigmoid()`, `nn.Tanh()`, `nn.LeakyRelu(...)`

In [ ]:
class FlexNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.activation = activation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

# Den første linje er lavet for dig. Udfyld aktiveringen i de tre andre — resten er ens:
history = train(FlexNet(nn.ReLU()), X, y);  plt.plot(history, label="ReLU")
history = train(FlexNet(...), X, y);        plt.plot(history, label="Sigmoid")    # <-- udfyld med Sigmoid
history = train(FlexNet(...), X, y);        plt.plot(history, label="Tanh")       # <-- udfyld med Tanh
history = train(FlexNet(...), X, y);        plt.plot(history, label="LeakyReLU")  # <-- udfyld med LeakyReLU

plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 2.3
Vi skruer op for støjen og ser, hvornår netværket knækker.

Prøv `noise=0.3`, `0.5` og `0.8` i `make_moons`, gentræn hver gang og plot beslutningsgrænsen. Hvornår kan modellen ikke længere finde en meningsfuld grænse — og hvordan ser den ud, når den forsøger at lære rent kaos?

Hint: jo mere støj, jo mere blander de to måner sig. Se på, om grænsen bliver mere og mere kroget for at ramme enkeltpunkter.

In [ ]:
X_np2, y_np2 = make_moons(n_samples=400, noise=0.3, random_state=42)   # ← skru på noise
X2 = torch.tensor(X_np2, dtype=torch.float32)
y2 = torch.tensor(y_np2, dtype=torch.float32)

model_noise = MoonNet()
train(model_noise, X2, y2)
plot_decision_boundary(model_noise, X2, y2, title="noise = 0.3")

##### Opgave 2.4 (find fejlen)
Netværket nedenfor skal lave **regression**: forudsige $y \approx 2x + 1$ (værdier fra 1 til 11).

Men forudsigelserne klistrer fast lige over 1, og tabet er enormt. Kig på `forward` — der har sneget sig et lag ind, som IKKE hører hjemme i en regressionsmodel. Prøv at fjerne det og se forskellen.

Sigmoid presser alt ned i intervallet (0, 1) — skidt, når svaret skal kunne blive 11 (jf. opgave 1.8d).

Hint: hvilket lag i `forward` begrænser outputtet til at ligge mellem 0 og 1?

In [ ]:
x_reg = torch.linspace(0, 5, 100).reshape(-1, 1)
y_reg = 2 * x_reg.squeeze() + 1 + torch.randn(100) * 0.3

class RegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model_reg = RegNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_reg.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model_reg(x_reg).squeeze(), y_reg)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    plt.scatter(x_reg, y_reg, alpha=0.4)
    plt.plot(x_reg, model_reg(x_reg).detach(), color="crimson", linewidth=2)
plt.title(f"sluttab: {loss.item():.2f}")
plt.show()

##### Opgave 2.7
Vi ser, hvor lidt et netværk med kun **én skjult neuron** (2 → 1 → 1) kan.

Prøv at træne det på månerne og plot beslutningsgrænsen. Beskriv, hvad du ser — og forklar, hvorfor grænsen ser sådan ud, når én ReLU-neuron kun kan "knække" planen ét sted.

Hint: sammenlign med ReLU-udgaven af MoonNet ovenfor, der havde 16+16 neuroner og kunne tegne en flot bugtet grænse.

In [ ]:
class MiniNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 1)     # én enkelt, ensom neuron
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(1, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.layer2(self.activation(self.layer1(x))))

model_mini = MiniNet()
train(model_mini, X, y, epochs=2000)
plot_decision_boundary(model_mini, X, y, title="1 skjult neuron")

##### Opgave 2.8
Til sidst kobler vi historien sammen: sigmoid kom først (1980'erne), ReLU tog over (2010'erne).

Ud fra alt, hvad du har set i denne notebook: prøv at give mindst to grunde til, at ReLU i dag er standardvalget i de skjulte lag — og én situation, hvor sigmoid stadig er det rigtige valg.

Hint: tænk på gradienterne (hvem forsvinder ude i halerne?), og på hvornår du har brug for at få en sandsynlighed ud.

*(Tænk over det, og diskutér med din sidemand — I behøver ikke skrive svaret ned.)*

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere. De bygger videre på det, du allerede har lavet, og du kan tage dem i den rækkefølge, du vil.

##### Ekstra 1
Nu ser vi på gradienterne af alle fire funktioner samlet.

Prøv at plotte gradienterne af sigmoid, tanh, ReLU og leaky ReLU i ét fælles plot (genbrug autograd-tricket fra teorien — én funktion ad gangen, samme `x`).

Hvilken funktion har den "sundeste" gradient for store positive $z$ — og hvad med for store negative?

Hint: en "sund" gradient er tæt på 1, så signalet slipper igennem. Kig især på, hvor ReLU og leaky ReLU adskiller sig ude i den negative side.

In [ ]:
funktioner = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "ReLU": torch.relu,
    "leaky ReLU": nn.functional.leaky_relu,
}

for name, f in funktioner.items():
    x = torch.linspace(-5, 5, 200, requires_grad=True)
    f(x).sum().backward()
    plt.plot(x.detach(), x.grad, label=name)   # x.grad ER gradienten

plt.legend()
plt.title("Gradienter")
plt.grid(True)
plt.show()

##### Ekstra 2
Nu ser du vanishing gradients live.

Skabelonen bygger et **6-lags** netværk, hvor aktiveringen kan vælges. Sigmoid-udgaven er trænet
og plottet for dig — du skal kun tilføje ReLU-udgaven ved at udfylde aktiveringen i den nederste linje.

Hvad ser du — og hvad har det med opgave 1.4 at gøre?

Hint: sigmoids gradient er højst 0,25. Ganget gennem seks lag bliver den forsvindende lille, og de forreste lag lærer knap nok noget.

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.layer = nn.ModuleList([nn.Linear(2, 16)] +
                                 [nn.Linear(16, 16) for i in range(4)] +
                                 [nn.Linear(16, 1)])
        self.activation = activation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layer[:-1]:
            x = self.activation(layer(x))
        return self.sigmoid(self.layer[-1](x))

# Sigmoid-udgaven er lavet for dig:
history_sig = train(DeepNet(nn.Sigmoid()), X, y, epochs=2000)
plt.plot(history_sig, label="6 lag, sigmoid overalt")

# Tilføj ReLU-udgaven — kun aktiveringen mangler:
history_relu = train(DeepNet(...), X, y, epochs=2000)   # <-- udfyld: nn.ReLU()
plt.plot(history_relu, label="6 lag, ReLU overalt")

plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Ekstra 3
Nu prøver vi klassifikation med **3 klasser**: `make_blobs` laver tre klumper af punkter.

Ved flere klasser skal netværket give **3 tal ud** (ét point pr. klasse), og tabsfunktionen skal være `nn.CrossEntropyLoss`. I skabelonen er alt fyldt ud på nær ét: prøv at udfylde tabsfunktionen.

**Vigtig detalje** (du får brug for den i notebook 4): `nn.CrossEntropyLoss` vil have de **rå point** — den kører selv softmax indeni! Derfor har modellen INGEN aktiveringsfunktion til sidst, og målene `y` skal være hele tal (klassenumre), ikke kommatal.

Hint: tabsfunktionen til flere klasser oprettes med `nn.CrossEntropyLoss()` — husk parenteserne.

In [ ]:
X_np3, y_np3 = make_blobs(n_samples=450, centers=3, cluster_std=1.2, random_state=42)
X3 = torch.tensor(X_np3, dtype=torch.float32)
y3 = torch.tensor(y_np3, dtype=torch.long)     # klassenumre 0, 1, 2 — hele tal!

class ThreeClassNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)          # 3 klasser ud -> 3 point (ét pr. klasse)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))   # rå point — ingen softmax her!

model3 = ThreeClassNet()
loss_fn = ...                              # ← udfyld: tabsfunktionen til flere klasser, nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model3.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model3(X3), y3)         # y3 er klassenumrene 0/1/2
    loss.backward()
    optimizer.step()

print("sluttab:", round(loss.item(), 4))
plot_decision_boundary(model3, X3, y3, title="3 klasser")